# Semantic Embedding Pipeline for Search-Query Terms

**Purpose:** Generate dense semantic embeddings for a bank of short search-query
phrases (single words, brand names, multi-word queries) using
`sentence-transformers/all-mpnet-base-v2`. These embeddings serve as the
**semantic baseline** to compare against SAX-based temporal clusters.

**Scope of this notebook:**
1. Environment setup
2. Model loading with key hyperparameters exposed
3. Data loading stub for the real 1203-term word bank
4. Text preprocessing (underscore-joined terms → natural phrases)
5. Full-batch embedding generation + save to disk

## 1. Environment Setup

Install and import required libraries. `sentence-transformers` wraps the
underlying HuggingFace `transformers` model and handles tokenization,
pooling, and normalization for us.

In [1]:
# Uncomment if running in a fresh environment (e.g., Colab)
# %pip install -U sentence-transformers torch pandas numpy

import os
import re
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

print(f"torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available:  {getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available()}")

c:\Python\CSUREMM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch version: 2.12.1+cpu
CUDA available: False
MPS available:  False


## 2 Model Loading

We use **`all-mpnet-base-v2`**, the strongest general-purpose model in the
`sentence-transformers` library for short-text semantic similarity. It is
well-suited to this dataset because it handles:

- single words (`"walmart"`, `"shakira"`)
- short colloquial phrases (`"2024 us election results"`)
- proper nouns / brand names not well covered by static word embeddings

without requiring domain-specific fine-tuning.

In [2]:
# --- CRITICAL HYPERPARAMETER: model name configuration ---
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

model = SentenceTransformer(MODEL_NAME)

# --- CRITICAL HYPERPARAMETER: maximum sequence length ---
# all-mpnet-base-v2 defaults to 384 tokens. Our inputs are short phrases
# (rarely more than 8-10 tokens), so we cap this low to save memory/compute
# without any risk of truncating real inputs.
model.max_seq_length = 64

print(model)
print(f"Max sequence length set to: {model.max_seq_length}")
print(f"Embedding dimensionality:   {model.get_sentence_embedding_dimension()}")

c:\Python\CSUREMM\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\20930\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1267.77it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'MPNetModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)
Max sequence length set to: 64
Embedding dimensionality:   768


C:\Users\20930\AppData\Local\Temp\ipykernel_47304\2564802216.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimensionality:   {model.get_sentence_embedding_dimension()}")


## 3. Data Loading — Real Word Bank

Loads the actual 1203-term word bank. Terms are stored one per line, with
multi-word queries joined by underscores (e.g., `2024_election_results`,
`stock_market_crash`).

In [4]:
DATA_PATH = r"C:\Python\CSUREMM\data\processed_gt\1203words.txt"

def load_terms_from_txt(path: str) -> list[str]:
    """Load one term per line from a plain-text file, skipping blanks."""
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def load_terms_from_csv(path: str, column: str = "term") -> list[str]:
    """Alternative loader stub if your word bank is a CSV with a term column."""
    df = pd.read_csv(path)
    return df[column].dropna().astype(str).tolist()

if os.path.exists(DATA_PATH):
    raw_terms = load_terms_from_txt(DATA_PATH)
else:
    print(f"[WARNING] '{DATA_PATH}' not found — falling back to mock_terms for demonstration.")
    raw_terms = mock_terms

print(f"Loaded {len(raw_terms)} raw terms")
print(raw_terms[:10])

Loaded 1203 raw terms
['2016_election_results', '2020_election_map', '2020_election_results', '2020_electoral_map', '2022_election', '2022_election_results', '2024_election', '2024_election_results', '2024_presidential_election', '2024_united_states_elections']


## 4. Text Preprocessing

Raw terms use underscores as word separators (an artifact of the Google
Trends export). We convert these into natural, space-separated phrases
before embedding — `all-mpnet-base-v2` was trained on natural language, not
underscore-joined tokens, so this step meaningfully improves embedding
quality.

We also strip stray punctuation artifacts (e.g., mojibake like `ame╠ürica`)
minimally, without altering legitimate characters (accents, ampersands,
apostrophes), since these can carry semantic meaning (`s&p_500`, `father's_day`).

In [5]:
def preprocess_term(term: str) -> str:
    """
    Convert an underscore-joined Trends query into a natural phrase
    suitable for sentence-embedding models.

    Steps:
      1. Replace underscores with spaces.
      2. Collapse repeated whitespace.
      3. Strip leading/trailing whitespace.
    Note: casing, accents, and punctuation (&, ', .) are intentionally
    preserved since they can carry semantic signal (e.g., "s&p 500").
    """
    cleaned = term.replace("_", " ")
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

processed_terms = [preprocess_term(t) for t in raw_terms]

# Show a few before/after examples
preview = pd.DataFrame({"raw": raw_terms[:8], "processed": processed_terms[:8]})
preview

,raw,processed
0,2016_election_results,2016 election results
1,2020_election_map,2020 election map
2,2020_election_results,2020 election results
3,2020_electoral_map,2020 electoral map
4,2022_election,2022 election
5,2022_election_results,2022 election results
6,2024_election,2024 election
7,2024_election_results,2024 election results


## 5. Full Batch Embedding Generation

Encode the complete, preprocessed word bank in a single batched call.
`batch_size` is exposed as a tunable hyperparameter.

In [6]:
BATCH_SIZE = 32

embeddings = model.encode(
    processed_terms,
    batch_size=BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print(f"Final embedding matrix shape: {embeddings.shape}")  # (n_terms, 768)
assert embeddings.shape[0] == len(processed_terms)

Batches: 100%|██████████| 38/38 [00:08<00:00,  4.71it/s]

Final embedding matrix shape: (1203, 768)


## 6. Save Embeddings

Persist both the embedding matrix and the term list (in matching row order)
so downstream clustering / distance-comparison notebooks can load them
directly.

In [7]:
OUTPUT_DIR = r"C:\Python\CSUREMM\output\bert"
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.save(os.path.join(OUTPUT_DIR, "mpnet_embeddings.npy"), embeddings)

terms_df = pd.DataFrame({"raw_term": raw_terms, "processed_term": processed_terms})
terms_df.to_csv(os.path.join(OUTPUT_DIR, "terms_index.csv"), index=False)

print(f"Saved embeddings to: {os.path.join(OUTPUT_DIR, 'mpnet_embeddings.npy')}")
print(f"Saved term index to: {os.path.join(OUTPUT_DIR, 'terms_index.csv')}")
print(f"Row order in terms_index.csv matches row order of the embedding matrix.")

Saved embeddings to: C:\Python\CSUREMM\output\bert\mpnet_embeddings.npy
Saved term index to: C:\Python\CSUREMM\output\bert\terms_index.csv
Row order in terms_index.csv matches row order of the embedding matrix.


In [13]:
import os
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
import plotly.express as px

# 1. Compress the 768-dimensional embeddings to 2D
tsne = TSNE(n_components=2, perplexity=2, random_state=42, init='pca', learning_rate='auto')
embeddings_2d = tsne.fit_transform(embeddings)

# 2. Apply jitter to prevent markers from stacking directly on top of each other
np.random.seed(42)
jitter_x = np.random.normal(0, 0.15, size=embeddings_2d.shape[0])
jitter_y = np.random.normal(0, 0.15, size=embeddings_2d.shape[0])

# 3. Build the clean DataFrame
inspect_df = pd.DataFrame({
    "raw_term": raw_terms,
    "x": embeddings_2d[:, 0] + jitter_x,
    "y": embeddings_2d[:, 1] + jitter_y
})

# 4. Create the scatter plot (REMOVED text="raw_term" to hide on-screen labels)
fig = px.scatter(
    inspect_df, 
    x="x", 
    y="y", 
    hover_name="raw_term",     # The word ONLY shows up in a crisp tooltip when hovering
    title="BERT Semantic Map (Hover over dots to reveal words)",
    template="plotly_white"    
)

# 5. Fine-tune marker style to make individual data points distinct and clean
fig.update_traces(
    marker=dict(
        size=12,               # Slightly larger dots since there is no text blocking them
        opacity=0.85, 
        line=dict(width=1.5, color='SlateGrey')
    )
)

# 6. Set a generous workspace and remove grid clutter
fig.update_layout(
    width=1100,               
    height=750,               
    hoverlabel=dict(
        bgcolor="white", 
        font_size=14, 
        font_family="Arial",
        bordercolor="SlateGrey"
    )
)

# 7. Save it as an interactive file
output_html = os.path.join(OUTPUT_DIR, "bert_map.html")
fig.write_html(output_html)
print(f"Saved text-hidden interactive inspection map to: {output_html}")


Saved text-hidden interactive inspection map to: C:\Python\CSUREMM\output\bert\bert_map.html
